# Re-ranker training

Trains the optional learned decision for the localizer's stage two, as required
by the submission FAQ when deep learning methods are used. The network re-ranks
the degenerate correlation candidates from their deviation fields; the default
pipeline does not use it unless the weights path is set, and inference is pure
numpy from the exported npz file.

Prerequisites: the training dataset and harvest, produced by
`scripts/generate_dataset.py` and `scripts/harvest_reranker_data.py`,
and torch installed via `requirements_train.txt`.


In [ ]:
import subprocess, sys
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PY = sys.executable
print('repository root:', REPO)


## 1. Generate training pairs

240 mixed pairs, seed 77, disjoint from every evaluation dataset.


In [ ]:
subprocess.run([PY, 'scripts/generate_dataset.py', '--style', 'mixed',
                '--num', '240', '--out', 'data/reranker_train240',
                '--seed', '77'], cwd=REPO, check=True)


## 2. Harvest candidate pools

Runs the classical pipeline on every pair and keeps the degenerate ones.


In [ ]:
subprocess.run([PY, 'scripts/harvest_reranker_data.py',
                '--datasets', 'data/reranker_train240',
                '--out', 'data/reranker_harvest'], cwd=REPO, check=True)


## 3. Train, calibrate and export

Within pair softmax with a learnable null class, dihedral augmentation,
abstention threshold calibrated on the validation split, weights exported
to `models/reranker.npz` with a numpy parity check. The run is recorded
with plots under `experiments/`.


In [ ]:
subprocess.run([PY, 'scripts/train_reranker.py',
                '--data', 'data/reranker_harvest',
                '--epochs', '40'], cwd=REPO, check=True)


## 4. Inspect the recorded run


In [ ]:
import json
run = sorted((REPO / 'experiments').glob('*_reranker_training'))[-1]
cfg = json.loads((run / 'config.json').read_text())
print('validation pair accuracy:', cfg['final_val_acc'])
print('calibrated abstention threshold:', cfg['tau'])
print('numpy torch parity:', cfg['numpy_torch_max_diff'])
from IPython.display import Image, display
display(Image(str(run / 'loss_curve.png')))
display(Image(str(run / 'val_accuracy.png')))


## 5. Evaluate with the re-ranker enabled

Compares against the recorded classical runs on the same datasets.


In [ ]:
subprocess.run([PY, 'scripts/evaluate.py', '--dataset', 'data/train40_v2',
                '--name', 'reranker_sem', '--reranker'], cwd=REPO, check=True)
subprocess.run([PY, 'scripts/evaluate.py', '--dataset', 'data/stress30',
                '--name', 'reranker_stress', '--reranker'], cwd=REPO, check=True)
